# Instruction Dataset Readiness

Fine-tuning starts before training. A model can only learn behavior that is visible in its examples, so this notebook focuses on data shape, validation, formatting, and a held-out evaluation split before any optimizer runs.

Examples can teach repeated style, refusal behavior, and formatting conventions when prompting alone is not enough.


## Learning Objectives

By the end of this notebook, you should be able to:

- Distinguish training examples from held-out evaluation cases and check prompt overlap.
- Validate instruction records and their links to approved reference material.
- Serialize the same example in completion and chat-message formats.


## From Raw Material to Training Data

```mermaid
flowchart LR
    A["Raw domain material"] --> B["Instruction examples"]
    B --> C["Schema validation"]
    C --> D["Completion format"]
    C --> E["Chat-message format"]
    C --> F["Held-out evaluation cases"]
```


## Supplied Data

We use three local files: `support_instruction_examples.jsonl` for supervised examples, `evaluation_cases.json` for held-out prompts, and `reference_material.json` for policy evidence used in later faithfulness checks.

| Term | Meaning in this repository |
| --- | --- |
| Instruction | The task or user request shown to the model. |
| Response | The target continuation the model should learn to produce. |
| Training example | An instruction-response pair used to update adapter weights. |
| Evaluation case | A held-out task used to test behavior after training. |
| Reference material | Source text used to verify whether an answer is faithful. |

The evaluation cases may cover the same policy domains as training, but they must not duplicate the training prompts. Otherwise, apparent improvement could be memorization rather than generalization.


<div align="center">
  <img src="assets/01-dolly-dataset-card.png" alt="Dataset card for databricks-dolly-15k" width="1000" style="max-width: 100%; height: auto;" />
  <p><em>Dataset card for databricks-dolly-15k, documenting an instruction dataset with the same instruction-response shape used here. Source: <a href="https://huggingface.co/datasets/databricks/databricks-dolly-15k" target="_blank" rel="noopener noreferrer">Hugging Face Hub, databricks/databricks-dolly-15k</a></em></p>
</div>


In [ ]:
import json
from pathlib import Path

DATA_DIR = Path("data")
# JSON Lines stores one complete training record on each line.
training_records = [
    json.loads(line)
    for line in (DATA_DIR / "support_instruction_examples.jsonl")
    .read_text()
    .splitlines()
]
evaluation_cases = json.loads((DATA_DIR / "evaluation_cases.json").read_text())
reference_material = json.loads((DATA_DIR / "reference_material.json").read_text())

len(training_records), len(evaluation_cases), len(reference_material)

In [ ]:
training_records[:2]

### Exercise: Check Exact Prompt Leakage

Find any instruction strings that appear in both the training records and held-out evaluation cases.


In [ ]:
# @TODO: Compute the intersection of training and evaluation instructions.


<details>
<summary>Show Answer</summary>

```python
training_instructions = {
    record["instruction"].strip().casefold() for record in training_records
}
evaluation_instructions = {
    case["instruction"].strip().casefold() for case in evaluation_cases
}
exact_prompt_leakage = sorted(training_instructions & evaluation_instructions)
assert not exact_prompt_leakage, "Training and evaluation prompts overlap."
exact_prompt_leakage
```

</details>


An empty result rules out exact prompt duplication. It does not rule out subtler leakage, such as nearly identical wording or copying an evaluation answer into training data, so dataset review still requires human judgment.


## Schema Validation

Validation catches broken records before they become hard-to-debug training failures. We use Pydantic because it gives clear errors and readable model definitions.

Schema validation answers structural questions: are required fields present, are values the expected types, and is text long enough to be plausible? It cannot decide whether an answer is correct, safe, or well written. Those semantic checks appear later as evidence validation and evaluation.


In [ ]:
from pydantic import BaseModel, Field


class TrainingExample(BaseModel):
    # One support-answer example used for supervised fine-tuning.
    id: str
    split: str = Field(pattern="^train$")
    reference_ids: list[str] = Field(min_length=1)
    instruction: str = Field(min_length=20)
    response: str = Field(min_length=20)


class EvaluationCase(BaseModel):
    # One held-out case used after training.
    id: str
    instruction: str = Field(min_length=20)
    reference_ids: list[str] = Field(min_length=1)
    ideal_answer: str = Field(min_length=20)

In [ ]:
validated_training = [
    TrainingExample.model_validate(record) for record in training_records
]
validated_evaluation = [
    EvaluationCase.model_validate(case) for case in evaluation_cases
]
reference_by_id = {item["id"]: item for item in reference_material}
assert len(reference_by_id) == len(reference_material), "Duplicate reference IDs."
assert len({item.id for item in validated_training}) == len(validated_training)
assert len({item.id for item in validated_evaluation}) == len(validated_evaluation)
validated_training[0]

### Exercise: Validate Reference IDs

Collect any `reference_ids` from training or evaluation examples that are missing from `reference_by_id`.


In [ ]:
# @TODO: Collect missing reference IDs from both `validated_training` and `validated_evaluation`.


<details>
<summary>Show Answer</summary>

```python
missing_reference_ids = sorted(
    {
        reference_id
        for example in [*validated_training, *validated_evaluation]
        for reference_id in example.reference_ids
        if reference_id not in reference_by_id
    }
)
assert not missing_reference_ids, "Unknown policy reference IDs."
missing_reference_ids
```

</details>


An empty list means every training and evaluation example can be traced to available policy evidence. That trace is useful later when we test hallucination and source faithfulness.


## Completion Format

Completion-style fine-tuning turns each record into one string. The delimiter convention matters because the model learns to continue after `### Response:`. Training and inference must use the same template; changing markers later creates a prompt distribution shift.


In [ ]:
def to_completion_text(example: TrainingExample) -> str:
    # Use stable delimiters so training and inference share one pattern.
    return (
        f"### Instruction:\n{example.instruction}\n\n### Response:\n{example.response}"
    )


completion_rows = [to_completion_text(example) for example in validated_training]
print(completion_rows[0])

### Exercise: Inspect Delimiters

Count how many serialized rows contain both `### Instruction:` and `### Response:`.


In [ ]:
# @TODO: Count rows that contain both delimiters.


<details>
<summary>Show Answer</summary>

```python
delimiter_count = sum(
    "### Instruction:" in row and "### Response:" in row for row in completion_rows
)
delimiter_count
```

</details>


## Chat-Style Format

Chat-style records keep system, user, and assistant roles separate. This is the format used by many instruction-tuning datasets and provider fine-tuning APIs, and notebook 03 renders these roles with SmolLM2's native chat template.

A model-specific chat template eventually converts these role dictionaries back into tokens. Keeping roles explicit still matters because it prevents accidental ambiguity about which content is an instruction and which content is the desired answer.


In [ ]:
def to_chat_messages(example: TrainingExample) -> list[dict[str, str]]:
    # Role separation clarifies which text is instruction and which text is target.
    return [
        {"role": "system", "content": "You are a concise support assistant."},
        {"role": "user", "content": example.instruction},
        {"role": "assistant", "content": example.response},
    ]


chat_rows = [to_chat_messages(example) for example in validated_training]
chat_rows[0]

## Dataset Quality Signals

Small fine-tuning datasets are fragile. We inspect response length and reference coverage because these signals catch overly verbose targets, missing domains, and accidental imbalance.

Quality is not the same as quantity. A small set of consistent, correct examples can teach a narrow behavior, while duplicated or contradictory examples can reinforce the wrong pattern. Useful checks include domain coverage, target length, duplicated prompts, conflicting answers, and traceability to source material.


In [ ]:
from collections import Counter

response_lengths = [len(example.response.split()) for example in validated_training]
reference_counts = Counter(
    reference_id
    for example in validated_training
    for reference_id in example.reference_ids
)
{
    "records": len(validated_training),
    "min_response_words": min(response_lengths),
    "max_response_words": max(response_lengths),
    "reference_counts": dict(reference_counts),
}

### Exercise: Flag Long Responses

Return IDs of examples with responses longer than 25 words.


In [ ]:
# @TODO: Find examples with responses longer than 25 words.


<details>
<summary>Show Answer</summary>

```python
long_response_ids = [
    example.id for example in validated_training if len(example.response.split()) > 25
]
long_response_ids
```

</details>


## Held-Out Evaluation Cases

Held-out cases act like a tiny test suite. They should represent important behavior without being directly optimized during training. Reusing policy topics is acceptable because the goal is to generalize the policy; reusing the exact prompt or target answer would weaken the test.


In [ ]:
for case in validated_evaluation:
    print(f"{case.id}: {case.instruction}")

### Exercise: Explain the Split

Why can an evaluation case reference the same refund policy as training examples without being data leakage?


<details>
<summary>Show Answer</summary>

The policy is the domain knowledge the model should apply. The held-out case tests a new situation and wording, so it measures whether the learned behavior transfers instead of checking whether the model can repeat an exact training pair.

</details>


## Further Reading

- [How to Format Fine-Tuning Data](https://help.openai.com/en/articles/6811186-how-do-i-format-my-fine-tuning-data-for-the-openai-api)
- [Datasets Documentation](https://huggingface.co/docs/datasets/index)
- [Dataset Cards](https://huggingface.co/docs/hub/datasets-cards)